# StylistAI CLIP search experiment

This notebook preserves the earlier image/text semantic-search prototype. It loads product metadata and precomputed image embeddings from Google Drive, encodes a text query with OpenAI CLIP ViT-B/32, ranks cosine similarities, and exposes results through a small FastAPI endpoint.

**This is a historical/optional experiment, not the recommendation path used by the current StylistAI frontend.** The current app uses the local rule-based `/api/recommendations` endpoint; CatVTON virtual try-on is documented separately in `catvton_colab.ipynb`.

All saved outputs and execution counts were cleared before publishing. The CLIP API key must be entered through a hidden prompt when running its server cell. Do not save the resulting output or commit credentials. The notebook expects its embedding and product files to be available in the user's own Google Drive.


In [ ]:
!pip -q install transformers pillow

In [ ]:

!pip -q install fastapi uvicorn pyngrok


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/StylistAI")

embeddings = np.load(SAVE_DIR / "fashion_embeddings.npy")
df = pd.read_csv(SAVE_DIR / "fashion_products_matched.csv")

print("Embeddings:", embeddings.shape)
print("Products:", len(df))
print("Columns:", df.columns.tolist())


In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()

embeddings = embeddings / np.linalg.norm(
    embeddings, axis=1, keepdims=True
)

print("CLIP loaded:", device)
print("Embeddings:", embeddings.shape)
print("Products:", df.shape)

In [ ]:
print("embeddings exists:", "embeddings" in globals())
print("df exists:", "df" in globals())

In [ ]:
import getpass
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import hmac
import uvicorn
import threading
from fastapi.middleware.cors import CORSMiddleware



API_KEY = getpass.getpass("Enter the CLIP API key (input is hidden): ")
app = FastAPI(title="StylistAI CLIP Search")
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_methods=["*"],
    allow_headers=["*"],
)

class SearchRequest(BaseModel):
    prompt: str
    limit: int = 8

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/search")
def search_products(
    request: SearchRequest,
    authorization: str = Header(default="")
):
    if not hmac.compare_digest(
        authorization.strip(),
        f"Bearer {API_KEY}"
    ):
        raise HTTPException(status_code=401, detail="Unauthorized")

    prompt = request.prompt.strip()

    if not prompt or len(prompt) > 300:
        raise HTTPException(
            status_code=400,
            detail="Prompt must be 1-300 characters."
        )

    text_inputs = processor(
        text=[prompt],
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        features = model.get_text_features(**text_inputs)

        if hasattr(features, "pooler_output"):
            features = features.pooler_output

        features = features / features.norm(dim=-1, keepdim=True)

    scores = embeddings @ features.cpu().numpy()[0]
    indices = scores.argsort()[::-1][:max(1, min(request.limit, 20))]

    results = []

    for i in indices:
        row = df.iloc[int(i)]
        results.append({
            "id": str(row.get("id", "")),
            "image": str(row.get("image", "")),
            "productDisplayName": str(row.get("productDisplayName", "")),
            "articleType": str(row.get("articleType", "")),
            "baseColour": str(row.get("baseColour", "")),
            "score": round(float(scores[i]), 4)
        })

    return {"prompt": prompt, "recommendations": results}

print("CLIP API code loaded.")

In [ ]:
import threading
import uvicorn
import time

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="error"
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)
api_thread.start()

time.sleep(2)
print("CLIP API server started.")

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

print("Status:", response.status_code)
print("Response:", response.text)

In [ ]:
from pyngrok import ngrok
import requests

# Reuse the token already saved in Colab
ngrok.kill()

tunnel = ngrok.connect(8000, "http")
CLIP_API_URL = tunnel.public_url

print("CLIP API URL:", CLIP_API_URL)

response = requests.get(
    CLIP_API_URL + "/health",
    timeout=30
)
print("Health:", response.status_code, response.text)

In [ ]:
response = requests.post(
    CLIP_API_URL + "/search",
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    json={
        "prompt": "white casual sneakers for men",
        "limit": 5
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.text[:4000])

In [ ]:
response = requests.get(CLIP_API_URL + "/health", timeout=15)
print(response.status_code, response.text)

In [ ]:
!fuser -k 8000/tcp

In [ ]:
from pyngrok import ngrok
import getpass

token = getpass.getpass("Paste your REAL ngrok authtoken: ")
ngrok.set_auth_token(token)

print("ngrok authentication configured")

In [ ]:
import requests

try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print(response.status_code)
    print(response.text)
except Exception as e:
    print("CLIP server is not responding:", e)

In [ ]:
print("CLIP API URL:", CLIP_API_URL)

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

In [ ]:
!pip install -r requirements.txt

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
